# retarget_view.ipynb — Subject / Frame 단위 시각화

**Registration 을 먼저 제대로 확인**한 뒤, `SUBJECT` 와 `FRAME` 두 개만 바꾸면
해당 프레임의 **① MRI · ② Registration · ③ Retargeting 모델**을 한 화면에 그려줍니다.

순서: 설정 → 헬퍼 → build(모델·registration·body contour 로드) → **registration 점검** → 프레임 시각화.

전제: `datasets/MRI_SSFP_10fps/<subject>/{png, body_contour.npz}`,
`datasets/tongue_model/tongue_rest_m.obj`, `test/v8/<subject>/registration.csv` 준비됨.
repo 루트에서 실행하세요.

In [ ]:
%matplotlib inline
import os, csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
import imageio.v2 as imageio
from scipy.interpolate import RBFInterpolator
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
plt.rcParams["figure.dpi"] = 110

REPO = Path.cwd()
assert (REPO/"datasets").exists(), f"repo 루트에서 실행하세요 (현재: {REPO})"

# ================== 여기 두 개만 바꾸면 됨 ==================
SUBJECT = "Subject3"
FRAME   = 30          # 보고 싶은 MRI 프레임 번호 (image_{FRAME}.png 기준)
# =========================================================

# ---- 고정 파라미터 (필요시 조정) ----
DATA          = REPO/"datasets"
N             = 60      # 닫힌 contour 리샘플 점수(=RBF center 수)
RBF_LEN       = 18.0    # skinning 길이 스케일(mm)
SPATIAL_WIN   = 3       # 곡선 따라(닫힘) 스무딩
TEMPORAL_WIN  = 9       # 프레임축 Savitzky-Golay(홀수, 1=off)
TEMPORAL_POLY = 2
MM_PER_PX     = 1.164
H             = 256
REST          = 0       # rest 프레임 인덱스(all_fr 기준, 0=첫 프레임)
print("config:", SUBJECT, "FRAME", FRAME)

## 1. 헬퍼 — 지오메트리 & registration
`load_obj`, 모델 y=0 midsag full contour, 닫힌 contour 정렬/리샘플,
그리고 **registration**(image→model affine + landmark 잔차)을 한 번에 로드.

In [ ]:
def load_obj(path):
    V, F = [], []
    for L in open(path):
        t = L.split()
        if not t: continue
        if t[0] == "v": V.append([float(x) for x in t[1:4]])
        elif t[0] == "f": F.append([int(s.split("/")[0]) - 1 for s in t[1:4]])
    return np.array(V), np.array(F)

def align_resample(c, n):
    """닫힌 contour → tip(min x) 시작, dorsal-up 방향, 호길이 n점."""
    c = np.asarray(c, float)
    c = np.roll(c, -int(np.argmin(c[:, 0])), axis=0)
    if c[1, 1] < c[-1, 1]: c = np.roll(c[::-1], 1, axis=0)
    cc = np.vstack([c, c[:1]])
    d = np.r_[0, np.cumsum(np.hypot(np.diff(cc[:,0]), np.diff(cc[:,1])))]
    if d[-1] == 0: return None
    u = np.linspace(0, d[-1], n, endpoint=False)
    return np.column_stack([np.interp(u,d,cc[:,0]), np.interp(u,d,cc[:,1])])

def model_midsag_contour(V_mm, F, n):
    """tongue mesh 를 y=0 로 잘라 닫힌 midsag 외곽선 (x,z) mm, n점."""
    segs = []
    for tri in F:
        p = V_mm[tri]; s = p[:, 1]; pts = []
        for a, b in [(0,1),(1,2),(2,0)]:
            if (s[a] <= 0 < s[b]) or (s[b] <= 0 < s[a]):
                t = s[a]/(s[a]-s[b]); pts.append((p[a]+t*(p[b]-p[a]))[[0,2]])
        if len(pts) == 2: segs.append((pts[0], pts[1]))
    used=[False]*len(segs); poly=[np.array(segs[0][0]), np.array(segs[0][1])]; used[0]=True
    for _ in range(len(segs)):
        cur=poly[-1]; nxt=None
        for i,sg in enumerate(segs):
            if used[i]: continue
            a,b=np.array(sg[0]),np.array(sg[1])
            if np.hypot(*(a-cur))<0.5: nxt=(i,b); break
            if np.hypot(*(b-cur))<0.5: nxt=(i,a); break
        if nxt is None: break
        used[nxt[0]]=True; poly.append(nxt[1])
    return align_resample(np.array(poly), n)

def load_registration(reg_csv):
    """registration.csv → landmark(image/model), image→model affine, landmark 잔차."""
    labels, img, mod = [], [], []
    for r in csv.DictReader(open(reg_csv)):
        labels.append(r["label"])
        img.append([float(r["imageX"]), float(r["imageY"])])
        mod.append([float(r["modelX"]), float(r["modelZ"])])
    img = np.array(img); mod = np.array(mod)
    A, *_ = np.linalg.lstsq(np.column_stack([img, np.ones(len(img))]), mod, rcond=None)
    to_model = lambda xy: np.column_stack([np.asarray(xy, float),
                                           np.ones(len(np.asarray(xy)))]) @ A
    resid = np.linalg.norm(to_model(img) - mod, axis=1)     # landmark별 잔차 mm
    return dict(labels=labels, img=img, mod=mod, A=A, to_model=to_model,
                resid=resid, rms=float(np.sqrt((resid**2).mean())))
print("helpers ready")

## 2. build — 모델·registration·body contour 로드
모델 full contour, registration affine, 그리고 body contour 를 model 공간으로 옮겨
`delta`(rest 대비 변형)와 프레임별 retarget mesh 를 준비합니다.
`SUBJECT` 을 바꾸면 이 셀부터 다시 실행하세요.

In [ ]:
sub = DATA/"MRI_SSFP_10fps"/SUBJECT
V, F = load_obj(str(DATA/"tongue_model"/"tongue_rest_m.obj")); V = V*1000.0; V[:,0] += 2.0  # mm
mc = model_midsag_contour(V, F, N)                              # (N,2) 모델 rest full contour

REG = load_registration(str(REPO/"test"/"v8"/SUBJECT/"registration.csv"))
to_model = REG["to_model"]

npz = np.load(str(sub/"body_contour.npz"))
fids = [int(x) for x in npz["frame_ids"]]
all_fr = list(range(min(fids), max(fids)+1))

def img_to_xy(rc):                                             # (row,col) → image-mm(x, y-up)
    return np.column_stack([rc[:,1]*MM_PER_PX, (H-1-rc[:,0])*MM_PER_PX])

def body_model(mi):
    key = f"resamp_f{mi}"
    if key not in npz.files: return None
    return align_resample(to_model(img_to_xy(npz[key])), N)   # model (x,z) mm, N점

stack = np.full((len(all_fr), N, 2), np.nan)
for t, mi in enumerate(all_fr):
    b = body_model(mi)
    if b is not None: stack[t] = b
for c in range(2):                                            # 결측 프레임 시간보간
    for j in range(N):
        col = stack[:,j,c]; ok = ~np.isnan(col)
        stack[:,j,c] = np.interp(np.arange(len(all_fr)), np.where(ok)[0], col[ok])

delta = stack - stack[REST]
if SPATIAL_WIN > 1: delta = uniform_filter1d(delta, SPATIAL_WIN, axis=1, mode="wrap")
if TEMPORAL_WIN > 1 and len(all_fr) >= TEMPORAL_WIN:
    delta = savgol_filter(delta, TEMPORAL_WIN, TEMPORAL_POLY, axis=0, mode="interp")
delta = delta - delta[REST]

Vxz = V[:,[0,2]]
def retarget(k):
    rbf = RBFInterpolator(mc, delta[k], kernel="gaussian", epsilon=1.0/RBF_LEN,
                          degree=-1, smoothing=1e-3)
    dxz = rbf(Vxz); Vd = V.copy(); Vd[:,0]+=dxz[:,0]; Vd[:,2]+=dxz[:,1]
    return Vd

deformed = [retarget(k) for k in range(len(all_fr))]          # 전 프레임(축·색 스케일 공용)
allv = np.vstack(deformed)
XL=(allv[:,0].min(),allv[:,0].max()); YL=(V[:,1].min(),V[:,1].max()); ZL=(allv[:,2].min(),allv[:,2].max())
gmax = max(np.linalg.norm(deformed[k]-V,axis=1).max() for k in range(len(all_fr))) or 1.0
rest_mid = V[np.abs(V[:,1])<6.0]
print(f"[build] {SUBJECT}: frames {all_fr[0]}..{all_fr[-1]} ({len(all_fr)})  |  reg RMS={REG['rms']:.2f}mm")

## 3. Registration 점검 (먼저 확인!)
Retargeting 을 보기 전에 **registration 이 제대로 됐는지** 먼저 확인합니다.

- **왼쪽**: MRI 위에 registration landmark(tip/dorsum/root)가 혀의 올바른 위치에 찍혔는지.
- **오른쪽**: image→model affine 으로 옮긴 landmark(빨강 ×)가 모델 landmark(검정 ○)와 겹치는지,
  그리고 MRI body 외곽선(빨강 점선)이 모델 rest 외곽선(초록)과 합리적으로 정렬되는지.

> 참고: landmark 가 3개면 2D affine(6 dof)이 정확히 결정돼 **RMS 는 항상 0** 입니다.
> 즉 RMS 는 landmark 가 4개 이상일 때만 의미가 있고, 3개일 땐 **landmark 위치와 외곽선 정렬을 눈으로** 확인하세요.

In [ ]:
def landmark_px(imgpt):
    """image-mm(x, y-up) → png 픽셀(col, row)."""
    return imgpt[:,0]/MM_PER_PX, (H-1) - imgpt[:,1]/MM_PER_PX

def check_registration(frame=None):
    frame = all_fr[REST] if frame is None else frame
    k = all_fr.index(frame)
    print(f"registration RMS = {REG['rms']:.3f} mm   (landmark {len(REG['labels'])}개)")
    for lab, e in zip(REG["labels"], REG["resid"]): print(f"  {lab:8s} 잔차 {e:.3f} mm")
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(10, 5))
    # MRI + landmarks
    g = imageio.imread(str(sub/"png"/f"image_{frame}.png")); a0.imshow(g, cmap="gray")
    key = f"contour_f{frame}"
    if key in npz.files:
        c = npz[key]; cc = np.vstack([c, c[:1]]); a0.plot(cc[:,1], cc[:,0], "-", c="lime", lw=1.4)
    lc, lr = landmark_px(REG["img"]); a0.scatter(lc, lr, c="red", marker="x", s=60, lw=2, zorder=5)
    for lab, x, y in zip(REG["labels"], lc, lr):
        a0.annotate(lab, (x, y), color="yellow", fontsize=9, xytext=(4,4), textcoords="offset points")
    a0.set_title(f"MRI f{frame} + landmarks", fontsize=10); a0.axis("off")
    # model space overlay
    a1.scatter(rest_mid[:,0], rest_mid[:,2], s=5, c="0.85", zorder=0)
    a1.plot(np.r_[mc[:,0],mc[:1,0]], np.r_[mc[:,1],mc[:1,1]], "g-", lw=1.3, label="model rest contour")
    bm = stack[k]
    a1.plot(np.r_[bm[:,0],bm[:1,0]], np.r_[bm[:,1],bm[:1,1]], "r--", lw=1.2, label="MRI body→model")
    mapped = to_model(REG["img"])
    a1.scatter(REG["mod"][:,0], REG["mod"][:,1], s=55, marker="o", edgecolors="k", facecolors="none", label="model landmark")
    a1.scatter(mapped[:,0], mapped[:,1], c="red", marker="x", s=45, label="mapped landmark")
    a1.set_aspect("equal", adjustable="box"); a1.set_xlim(XL); a1.set_ylim(ZL); a1.set_xticks([]); a1.set_yticks([])
    a1.set_title(f"registration check (RMS {REG['rms']:.2f}mm)", fontsize=10); a1.legend(fontsize=7, loc="lower left")
    fig.suptitle(f"{SUBJECT} — registration check", fontsize=11); fig.tight_layout(); plt.show()

check_registration(FRAME)

## 4. 프레임 시각화 — MRI · Registration · Retargeting
`FRAME` 프레임의 4패널: **① MRI+contour+landmark · ② registration(model 공간) · ③ retarget 3D mesh · ④ 변위(displacement)**.
`FRAME` 값만 바꾸고 이 셀을 다시 실행하면 됩니다. (맨 아래 슬라이더로 스크럽도 가능)

In [ ]:
def show(frame, save=None):
    assert frame in all_fr, f"frame {frame} 범위 밖 (사용 가능: {all_fr[0]}..{all_fr[-1]})"
    k = all_fr.index(frame); mi = frame
    Vd = deformed[k]; disp = np.linalg.norm(Vd - V, axis=1)
    fig = plt.figure(figsize=(19, 4.6))
    # 1) MRI + body contour + registration landmarks
    a0 = fig.add_subplot(1,4,1)
    g = imageio.imread(str(sub/"png"/f"image_{mi}.png")); a0.imshow(g, cmap="gray")
    key = f"contour_f{mi}"
    if key in npz.files:
        c = npz[key]; cc = np.vstack([c, c[:1]]); a0.plot(cc[:,1], cc[:,0], "-", c="lime", lw=1.5)
    lc, lr = landmark_px(REG["img"]); a0.scatter(lc, lr, c="red", marker="x", s=50, lw=1.8, zorder=5)
    for lab, x, y in zip(REG["labels"], lc, lr):
        a0.annotate(lab, (x, y), color="yellow", fontsize=8, xytext=(3,3), textcoords="offset points")
    a0.set_title(f"MRI f{mi} + landmarks", fontsize=9); a0.axis("off")
    # 2) registration (model space)
    a1 = fig.add_subplot(1,4,2); a1.scatter(rest_mid[:,0], rest_mid[:,2], s=5, c="0.85", zorder=0)
    mcd = mc + delta[k]
    a1.plot(np.r_[mc[:,0],mc[:1,0]], np.r_[mc[:,1],mc[:1,1]], "g-", lw=1.0, alpha=0.5, label="model rest")
    a1.plot(np.r_[mcd[:,0],mcd[:1,0]], np.r_[mcd[:,1],mcd[:1,1]], "b-", lw=1.4, label="model retarget")
    bm = stack[k]; a1.plot(np.r_[bm[:,0],bm[:1,0]], np.r_[bm[:,1],bm[:1,1]], "r--", lw=1.2, label="MRI body→model")
    mapped = to_model(REG["img"])
    a1.scatter(mapped[:,0], mapped[:,1], c="red", marker="x", s=35, zorder=5)
    a1.set_aspect("equal", adjustable="box"); a1.set_xlim(XL); a1.set_ylim(ZL); a1.set_xticks([]); a1.set_yticks([])
    a1.set_title("registration / model contour", fontsize=9); a1.legend(fontsize=6, loc="lower left")
    # 3) retargeted 3D mesh
    a2 = fig.add_subplot(1,4,3, projection="3d")
    a2.plot_trisurf(Vd[:,0], Vd[:,1], Vd[:,2], triangles=F, color="lightblue", alpha=0.9, linewidth=0.1, edgecolor="0.5")
    a2.set_xlim(XL); a2.set_ylim(YL); a2.set_zlim(ZL); a2.view_init(20,-70)
    a2.set_xticklabels([]); a2.set_yticklabels([]); a2.set_zticklabels([]); a2.set_title("retargeted 3D mesh", fontsize=9)
    # 4) displacement
    a3 = fig.add_subplot(1,4,4, projection="3d")
    tp = a3.plot_trisurf(Vd[:,0], Vd[:,1], Vd[:,2], triangles=F, cmap="viridis", linewidth=0.1, edgecolor="0.4")
    tp.set_array(disp[F].mean(1)); tp.set_clim(0, gmax)
    a3.set_xlim(XL); a3.set_ylim(YL); a3.set_zlim(ZL); a3.view_init(20,-70)
    a3.set_xticklabels([]); a3.set_yticklabels([]); a3.set_zticklabels([]); a3.set_title(f"displacement (max {disp.max():.1f}mm)", fontsize=9)
    fig.suptitle(f"{SUBJECT}  frame {mi}   |   registration RMS {REG['rms']:.2f}mm   |   disp max {disp.max():.1f}mm", fontsize=11)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.85, bottom=0.03, wspace=0.08)
    if save: fig.savefig(save, dpi=95); plt.close(fig); print("saved:", save)
    else: plt.show()

show(FRAME)

# (선택) 슬라이더로 프레임 스크럽
try:
    from ipywidgets import interact, IntSlider
    interact(lambda f: show(f), f=IntSlider(min=all_fr[0], max=all_fr[-1], step=1, value=FRAME))
except Exception as e:
    print("ipywidgets 없음 → show(프레임번호) 직접 호출:", e)